# EmpowerLens - Experiments 1-8, re-run so the results are comparable

Replaces the previous version of this notebook. Full reasoning in
[`docs/RERUN_PLAN.md`](../docs/RERUN_PLAN.md); the short version is below.

## Why the old results could not be used

| # | what was wrong | measured |
|---|---|---|
| 1 | The training data contained the test set | **396 rows** of `data/splits_combined/train.csv` also appear in Annotated val/test - **195 of 253 test rows, 77%** |
| 2 | Those leaked rows mostly carry the *wrong* label | the two corpora agree on only **36%** of them, so the model saw the test text with the wrong answer |
| 3 | Duplicate rows silently re-weighted training | 4,645 train rows, only **3,019 unique texts** |
| 4 | Every experiment sat a different exam | E2 scored each corpus on its own test set, E3-E8 on Combined, Month-1 on Annotated |
| 5 | Two copies of the script, wrong branch, scattered output | `src/` and `experiments/` both held it; notebook cloned `lumia-space`; results split across `results/` and `result_experiment/` |

## What is different now

- **Clean splits.** `src.make_splits_clean` writes *new* dirs, dropping any train
  row that appears in the yardstick's val/test plus internal duplicates. The
  frozen dirs are never edited, so old results stay traceable.
- **Two exams per run.** Every checkpoint is scored on its **home** test set
  (within-dataset performance) *and* on the **yardstick** -
  `data/splits/test.csv`, the same 253 human-annotated rows for every
  experiment. `transfer_gap = home - yardstick`.
- **One results root**, `results_RUN2/results_experiments/`, one naming scheme, one branch.
- **Same protocol everywhere** - same model, same 3 seeds, same epochs. Only the
  thing under test varies.

> **The rule this notebook enforces:** two numbers are comparable only if both
> come from the **yardstick** exam, on the **same task**, with `leaked = False`.
> Every results row carries all three fields.

## 0. Setup

**Kaggle:** Settings -> Accelerator **GPU T4**, Internet **On**.

`CUDA_VISIBLE_DEVICES=0` pins to one GPU - T4 x2 causes a cross-device deadlock
in this stack (documented in the project notes).

> ⚠️ **Run this cell exactly once per session.** It starts with `rm -rf`, so
> re-running it deletes every result and checkpoint trained so far. If you need
> to restart, begin from the next cell.

In [ ]:
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch, transformers, subprocess
# Provenance. A result that cannot be traced to a GPU, a commit and a library
# version cannot be defended when two runs disagree later.
print("CUDA        :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("commit      :", subprocess.run(["git", "rev-parse", "HEAD"],
                                      capture_output=True, text=True).stdout.strip())
print()
print("Record the GPU name with your results - determinism guarantees a re-run")
print("matches on the SAME hardware, not across different GPUs.")

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "experiments/experiments_flat_mentalroberta.py").exists():
    if (ROOT.parent / "experiments/experiments_flat_mentalroberta.py").exists():
        ROOT = ROOT.parent
        os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

SCRIPT = "experiments/experiments_flat_mentalroberta.py"
assert Path(SCRIPT).exists(), f"{SCRIPT} not found - is the branch right?"
PY = sys.executable

# ---- one place to change the protocol; every experiment below uses it ----
MODEL     = "mental/mental-roberta-base"
SEEDS     = "42,1337,2024"        # project convention: 3 seeds, mean +/- std
EPOCHS    = 8                     # was 4, never validated; 8 lets the peak appear
MAX_LEN   = 512                   # 2.4% truncation, vs 24.9% at 256
DETERM    = True                  # pin algorithms so the same seed reproduces
OUT_ROOT  = "results_RUN2/results_experiments"   # RUN2 = the post-fix rerun;
                                                # RUN1 is frozen history, never write there

YARDSTICK = "data/splits"                    # the fixed exam, never changes
ANNOTATED = "data/splits"
CODIPAS   = "data/splits_codipas_clean"           # built by the preflight cell
COMBINED  = "data/splits_combined_clean"          # built by the preflight cell
MATCHED   = "data/splits_codipas_transfer_matched"  # frozen; 2,024 rows, leak=0
COMB_MATCH = "data/splits_combined_matched"       # built by the preflight cell:
                                                  # combined downsampled to 2,024

DET = ["--deterministic"] if DETERM else []

print(f"repo   : {ROOT}")
print(f"script : {SCRIPT}")
print(f"model  : {MODEL} | seeds {SEEDS} | {EPOCHS} epochs | max_len {MAX_LEN}")
print(f"out    : {OUT_ROOT}/")


def run(cmd, label=""):
    # Stream a subprocess live so a long training run shows progress.
    if label:
        print(f"\n{'=' * 72}\n{label}\n{'=' * 72}")
    print("$", " ".join(str(c) for c in cmd), "\n")
    p = subprocess.Popen([str(c) for c in cmd], cwd=ROOT, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise SystemExit(f"FAILED (exit {p.returncode}): {' '.join(map(str, cmd))}")
    return p.returncode

## 1. Preflight - prove there is no leakage

**Nothing below this cell is worth running until this passes.** It rebuilds the
derived split dirs with the contamination removed, then re-audits its own output
and fails if anything survives.

`data/splits` (the yardstick) is never touched - it is the reference, and it has
no leak.

In [ ]:
# Build the clean dirs. --force is safe here: they are derived, not frozen,
# and fully reproducible from the sources plus this script.
run([PY, "-m", "src.make_splits_clean", "--force"], "Building leakage-free splits")

# Downsample combined to Annotated's exact row count. Without this arm,
# annotated_only (2,024 rows, human labels) vs combined_clean (2,623 rows, mixed
# labels) varies BOTH volume and label convention, and no result can say which
# caused what. Stratified on y_mc, fixed seed, val/test untouched.
run([PY, "-m", "src.make_splits_clean", "--source", COMBINED,
     "--dest", COMB_MATCH, "--match-to", ANNOTATED, "--force"],
    "Volume-matched combined arm")

# Now prove it. Exit code 1 means a leak survived somewhere.
print(f"\n{'=' * 72}\nAUDIT\n{'=' * 72}")
p = subprocess.run([PY, "-m", "src.make_splits_clean", "--check"], cwd=ROOT,
                   capture_output=True, text=True, encoding="utf-8", errors="replace")
print(p.stdout)

for d in (ANNOTATED, CODIPAS, COMBINED):
    man = Path(d) / "clean_manifest.json"
    n = len(pd.read_csv(Path(d) / "train.csv", encoding="utf-8-sig"))
    note = ""
    if man.exists():
        m = json.loads(man.read_text(encoding="utf-8"))
        note = (f"  (was {m['before']['train_rows']}, removed "
                f"{m['after']['rows_removed']} = {m['after']['removed_pct']}%)")
    print(f"{d:<32} train={n}{note}")

print("\nThe pre-fix dirs (data/splits_combined, data/splits_codipas_cls) still "
      "show a leak\nabove - that is expected and correct. They are kept unedited "
      "so older results\nstay traceable. Nothing below trains on them.")

## 2. Determinism check - prove a re-run gives the same answer

**Every error bar below depends on this passing.**

Setting a seed makes the *random* parts repeatable. It does not make the
*arithmetic* repeatable: a GPU adds thousands of numbers in whatever order its
threads finish, and floating-point addition is order-dependent. Summing the same
100,000 floats forwards, backwards and shuffled gives `-90.825134`,
`-90.825070`, `-90.825070`.

That compounded badly here. Experiments 6 and 7 in the previous suite were the
*same configuration* and disagreed by **0.047 macro-F1 at seed 42**, while
reporting a seed-to-seed SD of **0.007**. The published error bars were seven
times smaller than the actual run-to-run noise, so any difference under ~0.05 was
unreadable.

`--deterministic` pins the algorithm choice. `warn_only=True` means an op with no
deterministic kernel warns instead of crashing the run - which is why this
verification is not optional: it turns "determinism requested" into "determinism
demonstrated".

In [ ]:
run([PY, "-m", "src.determinism", "--check"], "Determinism self-check")
print("\nIf that says PASS, the same seed now reproduces bit-for-bit and "
      "mean +/- SD\nhonestly means 'sensitivity to seed'. If it says FAIL, stop "
      "and investigate -\nno +/- figure from this machine is trustworthy.")

## 3. Experiment 1, pass 1 - the data audit

No training, no GPU, seconds. **Run it first**: the class imbalance it reports is
the gate that decides whether Experiments 3, 4 and 5 are worth the GPU time.

Produces, per the spec:

- class frequencies for binary and 11-class
- per-label prevalence for multilabel, **split by train / val / test** so a
  skewed split shows up
- **label co-occurrence** - counts and Jaccard - which pairs of distortions the
  annotators put on the same row

The per-class precision/recall/F1 and confusion matrices need trained models, so
they come in pass 2 (section 6), after E6 has produced checkpoints. In the old
notebook that half was commented out and never ran at all.

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 1, "--splits", ANNOTATED, "--out", f"{OUT_ROOT}/exp1"],
    "E1 pass 1 - data / label audit (Annotated)")

for f in sorted(Path(f"{OUT_ROOT}/exp1").glob("exp1_*.csv")):
    print(f"\n--- {f.name} ---")
    display(pd.read_csv(f).round(3))

## 4. Experiment 8 - sequence length and truncation

No training. Cheap, and it decides `MAX_LEN` for everything else.

Reports median, 90th and 95th percentile token length, and the percentage of
samples truncated at each candidate `max_length`. A Longformer run only launches
if truncation at 512 exceeds the threshold.

This is what exposed the previous suite's biggest configuration error: the
cascade ran at `max_length 256`, where **24.9% of test rows truncate**, against
E6's 512 at **2.4%**. Part of the gap between those two experiments was lost
text, not architecture.

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 8, "--splits", ANNOTATED, "--model", MODEL,
     "--max-length", MAX_LEN, "--truncation-threshold-pct", 10,
     "--out", f"{OUT_ROOT}/exp8"] + DET,
    "E8 - sequence length / truncation (Annotated)")

## 5. Experiment 6 - multilabel per-label performance and thresholds

Per-label results, label-wise loss weighting, and threshold optimisation swept on
**val only** then frozen for test. Tuning thresholds on test would make the test
number meaningless.

Run before E2 because it is cheaper, it produces the checkpoints E1 pass 2 needs,
and it gives you a usable headline early in case the session is cut short.

~3 seeds x 8 epochs. Budget roughly 1-1.5 hr on a T4.

In [ ]:
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 6, "--task", "multilabel", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--loss", "weighted_bce",
     "--out", f"{OUT_ROOT}/exp6"] + DET,
    "E6 - multilabel per-label + thresholds (Annotated)")

In [ ]:
# Did 8 epochs turn out to be enough? Every run now saves its curve, so this is
# answerable instead of assumed. If the best epoch is consistently 8 of 8, the
# budget is still too small and every result here is under-trained.
rows = []
for ck in sorted(Path(f"{OUT_ROOT}/exp6/checkpoints").glob("*")):
    h = ck / "epoch_history.csv"
    if h.exists():
        d = pd.read_csv(h)
        best = d.loc[d["eval_macro_f1"].idxmax()]
        rows.append({"run": ck.name, "best_epoch": int(round(best["epoch"])),
                     "of": int(round(d["epoch"].max())),
                     "best_val_macro_f1": round(best["eval_macro_f1"], 4)})
if rows:
    e = pd.DataFrame(rows)
    display(e)
    at_limit = (e["best_epoch"] >= e["of"]).sum()
    if at_limit:
        print(f"{at_limit}/{len(e)} runs peaked at the LAST epoch - the budget is "
              f"probably too small.\nRaise EPOCHS and re-run before trusting these "
              f"numbers.")
    else:
        print(f"All {len(e)} runs peaked before the limit, so {EPOCHS} epochs was "
              f"enough.\nThat is now evidence rather than assumption.")

# --- Did the runs actually use the recipe you set in section 0? --------------
# Not paranoia. The recipe used to be dropped on the way into each per-seed
# subprocess: --epochs 8 became 4 and --deterministic vanished, silently, while
# the section 2 gate still printed PASS because it tests the flag on its own
# rather than inside a run. Eight hours of GPU later the numbers were unusable.
# meta.json records what each run really used, so check it here - after the
# first 40-minute section, not after the whole suite.
want = {"epochs": EPOCHS, "max_length": MAX_LEN, "model": MODEL}
drift = []
for ck in sorted(Path(f"{OUT_ROOT}/exp6/checkpoints").glob("*")):
    mp = ck / "meta.json"
    if not mp.exists():
        continue
    m = json.loads(mp.read_text(encoding="utf-8"))
    for k, v in want.items():
        if m.get(k) != v:
            drift.append(f"{ck.name}: {k} = {m.get(k)!r}, expected {v!r}")
    got_det = bool(m.get("determinism", {}).get("deterministic"))
    if got_det != DETERM:
        drift.append(f"{ck.name}: deterministic = {got_det}, expected {DETERM}")

if drift:
    print()
    print("RECIPE DRIFT - these runs did NOT use the settings from section 0:")
    for d in drift:
        print("   ", d)
    raise SystemExit("Stop and fix before spending more GPU time.")
elif rows:
    print()
    print(f"Recipe verified on {len(rows)} checkpoints: "
          f"{MODEL}, {EPOCHS} epochs, max_len {MAX_LEN}, deterministic={DETERM}.")

## 6. Experiment 1, pass 2 - the model half of the audit

Now that E6 has produced checkpoints, re-run E1 with them to get the part the
spec asks for and the old notebook never produced: **per-class precision /
recall / F1** and **confusion matrices**.

Confusion matrices exist for binary and multiclass only. Multilabel has no single
predicted class per row to cross-tabulate, so its equivalent is the per-label
table.

In [ ]:
ml_ckpts = sorted(Path(f"{OUT_ROOT}/exp6/checkpoints").glob("*"))
if not ml_ckpts:
    print("No E6 checkpoints yet - run section 5 first.")
else:
    run([PY, "-m", "experiments.experiments_flat_mentalroberta",
         "--experiment", 1, "--splits", ANNOTATED, "--out", f"{OUT_ROOT}/exp1",
         "--checkpoint-ml", str(ml_ckpts[0])],
        "E1 pass 2 - per-class tables + confusion matrices")

    for f in sorted(Path(f"{OUT_ROOT}/exp1").glob("exp1_*per_class*.csv")):
        print(f"\n--- {f.name} ---")
        display(pd.read_csv(f).round(3))

    from IPython.display import Image, display as _d
    for p in sorted(Path(f"{OUT_ROOT}/exp1").glob("confusion_*.png")):
        print(f"\n{p.name}")
        _d(Image(str(p)))

## 7. Experiment 2 - dataset ablation (the heaviest)

**The question:** does adding CODIPAS help, hurt, or do nothing?

**Five arms**, not three. The fourth is the one that actually isolates the
effect:

| arm | trains on | why |
|---|---|---|
| `annotated_only` | 2,024 human-annotated rows | the baseline |
| `codipas_clean` | 2,224 rule-labelled rows | does rule-labelled data transfer? |
| **`codipas_matched`** | **2,024 rule-labelled rows** | **same question, volume controlled** |
| **`combined_matched`** | **2,024 mixed rows** | **merging, volume controlled** |
| `combined_clean` | 2,623 rows | does merging help? |

`data/splits_codipas_transfer_matched` is downsampled to exactly 2,024 rows - the
same as Annotated - so a difference cannot be explained by "it just had more
data". Its test set **is** the yardstick, and I verified leak = 0.

**Every arm is scored on the same yardstick**, which the previous run did not do:
it scored each arm on its own test set, so the ranking may have been measuring
which test set was easiest rather than which training set was best.

**What we expect: merging hurts the fine-grained task.**
`docs/codipas_agreement.md` compared the two label sets over the 2,520 shared
texts:

| comparison | agreement | Cohen's κ |
|---|---|---|
| binary - distorted or not? | 66.5% | 0.321 (fair) |
| 11-class - which distortion? | 36.8% | **0.199 (slight)** |
| which type, among rows *both* call distorted | **27.5%** | — |

Even where both agree a distortion is present, they disagree about **which one
73% of the time**. Two schemes agreeing at κ = 0.199 are not labelling the same
thing, so this is not a "more data" experiment. **A clear negative result on a
fixed test set is a real finding**, and more defensible than a marginal win.

Each cell trains one arm (all 3 seeds) and frees the GPU before the next. Budget
~1-1.5 hr per arm; run them across sessions if needed, results accumulate.

In [ ]:
# 7a. Annotated only - the baseline. home == yardstick.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel", "--only-config", "annotated_only",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN,
     "--out", f"{OUT_ROOT}/exp2"] + DET,
    "E2a - Annotated only")

In [ ]:
# 7b. CODIPAS only, cleaned. The transfer arm: home is CODIPAS's test set,
# yardstick is Annotated's, and the gap between them is the headline.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel", "--only-config", "codipas_only",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN,
     "--out", f"{OUT_ROOT}/exp2"] + DET,
    "E2b - CODIPAS only (clean)")

In [ ]:
# 7c. CODIPAS, volume-matched to Annotated (2,024 rows) and already scored on the
# Annotated test set. Runs through the same codipas_only path with a different
# --codipas-splits, so nothing about the protocol changes except the data.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel", "--only-config", "codipas_only",
     "--annotated-splits", ANNOTATED, "--codipas-splits", MATCHED,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN,
     "--out", f"{OUT_ROOT}/exp2_matched"] + DET,
    "E2c - CODIPAS, volume-matched to Annotated")

In [ ]:
# 7d. Annotated + CODIPAS, cleaned.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel",
     "--only-config", "annotated_plus_codipas",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMBINED, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN,
     "--out", f"{OUT_ROOT}/exp2"] + DET,
    "E2d - Annotated + CODIPAS (clean)")

In [ ]:
# 7e. Combined, downsampled to Annotated's 2,024 rows. THIS is the arm that
# separates the two effects:
#   annotated_only  vs combined_matched -> label convention, volume FIXED
#   combined_matched vs combined_clean  -> volume, convention FIXED
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 2, "--task", "multilabel",
     "--only-config", "annotated_plus_codipas",
     "--annotated-splits", ANNOTATED, "--codipas-splits", CODIPAS,
     "--combined-splits", COMB_MATCH, "--model", MODEL, "--seeds", SEEDS,
     "--epochs", EPOCHS, "--max-length", MAX_LEN,
     "--out", f"{OUT_ROOT}/exp2_combmatched"] + DET,
    "E2e - Combined, volume-matched to Annotated (2,024 rows)")

In [ ]:
# 7f. Score every arm on BOTH exams. This is what makes E2 readable - without
# the yardstick column the four arms sat four different exams.
# E2 writes each arm to its own subdir (exp2/annotated_only/checkpoints/...),
# unlike E6 which writes exp6/checkpoints/ directly. This used to glob only the
# flat layout, found nothing for E2, and skipped the loop in silence - so the
# heaviest experiment in the suite produced no yardstick column at all, which is
# the one thing E2 was rebuilt to produce. Cover both layouts, and assert, so a
# future layout change fails loudly instead of quietly emptying the table.
def find_checkpoints(root):
    root = Path(root)
    seen, out = set(), []
    for pattern in ("checkpoints/*", "*/checkpoints/*"):
        for ck in sorted(root.glob(pattern)):
            if ck.is_dir() and (ck / "meta.json").exists() and ck not in seen:
                seen.add(ck); out.append(ck)
    return out

for root in (f"{OUT_ROOT}/exp2", f"{OUT_ROOT}/exp2_matched",
             f"{OUT_ROOT}/exp2_combmatched"):
    cks = find_checkpoints(root)
    assert cks, (f"No checkpoints under {root}/ - E2 cannot be scored. "
                 f"Did the training cells above actually run?")
    print(f"{root}: {len(cks)} checkpoints")
    for ck in cks:
        run([PY, "-m", "src.eval_two_exams", "--checkpoint", ck,
             "--out", root, "--max-labels", 2, "--tag", ck.name])

frames = [pd.read_csv(f) for f in
          [Path(f"{OUT_ROOT}/exp2/two_exams.csv"),
           Path(f"{OUT_ROOT}/exp2_matched/two_exams.csv"),
           Path(f"{OUT_ROOT}/exp2_combmatched/two_exams.csv")] if f.exists()]
if frames:
    e2 = pd.concat(frames, ignore_index=True)
    display(e2.round(3))

In [ ]:
# The comparison E2 exists for: same exam, four training corpora.
# Per-seed AND mean +/- SD, as the spec asks - a mean that hides one wild seed
# is not a result.
if frames:
    yard = e2[e2["exam"] == "yardstick"]
    if not yard.empty:
        print("PER SEED (yardstick = data/splits/test.csv, same 253 rows for all "
              "arms):")
        display(yard.pivot_table(index="trained_on", columns="seed",
                                 values="macro_f1").round(3))
        print("\nMEAN +/- SD:")
        agg = yard.groupby("trained_on")["macro_f1"].agg(["mean", "std", "count"])
        agg.columns = ["macro_f1_mean", "macro_f1_sd", "seeds"]
        display(agg.round(3))
        print("\nIf annotated_only >= combined, merging the corpora costs "
              "accuracy -\nthe predicted result, and a reportable one.")
        print("If codipas_matched ~ codipas_clean, volume was not the driver; "
              "the\nlabel convention was.")

## 8. Experiments 3 -> 4 -> 5 - the imbalance chain

**Run in order.** Each reads the previous winner: E4 is only worth running if
E3's weighted CE did not close the gap, and E5 pairs sampling with whichever loss
won.

All three train on **Annotated**, so home == yardstick and they compare directly
to each other, to E6, and to the Month-1 baselines. Previously they pointed at
`data/splits_combined`, which contained 77% of its own test set.

**Skip this whole section if E1 showed the imbalance is mild** - that is what E1
is for.

In [ ]:
# E3 - plain CE vs class-weighted CE (multiclass, 11 classes)
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 3, "--task", "multiclass", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--out", f"{OUT_ROOT}/exp3"] + DET,
    "E3 - CE vs weighted CE (Annotated, multiclass)")

In [ ]:
# E4 - focal vs class-balanced. Only worth running if E3's gain was insufficient.
run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 4, "--task", "multiclass", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--gamma", 2.0, "--cb-beta", 0.999,
     "--out", f"{OUT_ROOT}/exp4"] + DET,
    "E4 - focal vs class-balanced (Annotated, multiclass)")

In [ ]:
# E5 - weighted sampling vs the best loss from E3/E4.
BEST_LOSS = "weighted_ce"      # <-- set this from the E3/E4 tables above

run([PY, "-m", "experiments.experiments_flat_mentalroberta",
     "--experiment", 5, "--task", "multiclass", "--splits", ANNOTATED,
     "--model", MODEL, "--seeds", SEEDS, "--epochs", EPOCHS,
     "--max-length", MAX_LEN, "--best-loss", BEST_LOSS,
     "--out", f"{OUT_ROOT}/exp5"] + DET,
    f"E5 - weighted sampling vs {BEST_LOSS} (Annotated, multiclass)")

## 9. Experiment 7 - flat vs cascade

**Not run here. Owned by Nayab**, who runs *both* arms in **one session on one
GPU** so they share hardware as well as configuration.

The gap being measured is tiny — the last comparison gave **+0.003**. Determinism
makes a re-run reproducible on the *same* machine; it does not make a T4 and a
P100 agree, and Kaggle hands out both. Splitting the arms across two people would
leave architecture and hardware varying together.

The flat multilabel number from Section 5 (E6) is a **replication check** for
that comparison, not the comparator itself.

The exact configuration both sides must match is pinned in
[`docs/E7_PROTOCOL.md`](../docs/E7_PROTOCOL.md). The essentials:

- `mental/mental-roberta-base`, `data/splits`, seeds 42/1337/2024
- **`max_length 512`** - the previous cascade ran at 256, truncating **24.9%** of
  test rows against E6's 2.4%, so that comparison was confounded
- **Stage 2 splits regenerated** from `data/splits` - the shipped
  `data/splits_stage2` was derived from the leaked `splits_combined`
- **determinism on** - the expected flat-vs-cascade gap is small, and the
  previous noise floor (~0.047) would have swallowed it

Three numbers reported separately: Stage 1 binary, Stage 2 multilabel *in
isolation*, and end-to-end. Stage 2's isolated score must never be quoted as a
cascade result - it is measured only on rows already known to be distorted, so it
hides every Stage 1 false negative.

In [ ]:
print(Path("docs/E7_PROTOCOL.md").read_text(encoding="utf-8")[:2000])

## 10. One comparable table

Collects every `two_exams.csv` under `results_RUN2/results_experiments/` and applies the
comparability rule: **yardstick exam, same task, `leaked = False`**. Anything
failing that is shown separately rather than silently mixed in.

In [ ]:
frames = []
for f in sorted(Path(OUT_ROOT).glob("exp*/two_exams.csv")):
    d = pd.read_csv(f)
    d.insert(0, "experiment", f.parent.name)
    frames.append(d)

if not frames:
    print("No two_exams.csv found yet - run the evaluation cells above.")
else:
    allr = pd.concat(frames, ignore_index=True)
    allr.to_csv(f"{OUT_ROOT}/all_runs_two_exams.csv", index=False)

    comparable = allr[(allr["exam"] == "yardstick") & (~allr["leaked"])]
    print(f"{len(comparable)} comparable rows of {len(allr)} total\n")

    for task, grp in comparable.groupby("task"):
        headline = {"binary": "positive_class_f1",
                    "multiclass": "macro_f1_10"}.get(task, "macro_f1")
        print(f"--- task: {task}  (yardstick = data/splits/test.csv) ---")
        piv = (grp.groupby(["experiment", "trained_on", "loss"])[headline]
                  .agg(["mean", "std", "count"]).round(3)
                  .sort_values("mean", ascending=False))
        piv.columns = [f"{headline}_mean", f"{headline}_sd", "seeds"]
        display(piv)

    excluded = allr[(allr["exam"] != "yardstick") | (allr["leaked"])]
    if not excluded.empty:
        print(f"\n{len(excluded)} rows NOT in the comparison above (home exam, "
              f"zero-shot, or leaked).\nShown for the transfer story only:")
        display(excluded[["experiment", "trained_on", "exam", "leaked",
                          "macro_f1", "micro_f1"]].round(3))

In [ ]:
# Transfer story: home vs yardstick for anything trained off-corpus.
allr = pd.read_csv(f"{OUT_ROOT}/all_runs_two_exams.csv")
tr = allr[(~allr["home_is_yardstick"]) & allr["exam"].isin(["home", "yardstick"])]
if tr.empty:
    print("Nothing trained off-corpus yet - run E2b/E2c.")
else:
    p = tr.pivot_table(index=["experiment", "trained_on", "seed"],
                       columns="exam", values="macro_f1")
    p["transfer_gap"] = p["home"] - p["yardstick"]
    display(p.round(3))
    print("\nhome      = did it learn its own corpus?")
    print("yardstick = does that carry over to the human-annotated task?")
    print("gap       = how much was corpus-specific.\n")
    print("The yardstick row is the CALIBRATED one: thresholds re-swept on the")
    print("yardstick's val, never its test. Applying CODIPAS-fitted thresholds to")
    print("Annotated data would understate transfer because the calibration is")
    print("wrong, not the model. The zero-shot row is in the excluded table above.")

## 11. Collect and zip

Copies everything into `/kaggle/working/` and packs one archive. Checkpoints are
excluded - each is ~500 MB and there are many.

**Run this before the session ends.** Kaggle reclaims the working directory on
teardown; anything not copied out is gone.

In [ ]:
import shutil, zipfile, datetime

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dest = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT
zip_path = dest / f"empowerlens_rerun_{stamp}.zip"

def add(zf, path, arc_root):
    path = Path(path)
    if path.is_file():
        zf.write(path, Path(arc_root) / path.name)
    elif path.is_dir():
        for f in sorted(path.rglob("*")):
            # Weights are huge and reproducible from the code + splits. The
            # epoch_history.csv inside each checkpoint dir is small and worth
            # keeping, so it is let through explicitly.
            if f.is_file() and ("checkpoints" not in f.parts
                                or f.name == "epoch_history.csv"
                                or f.name == "meta.json"):
                zf.write(f, Path(arc_root) / f.relative_to(path))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    add(zf, OUT_ROOT, OUT_ROOT)
    for doc in ("docs/RERUN_PLAN.md", "docs/E7_PROTOCOL.md"):
        if Path(doc).exists():
            add(zf, doc, "docs")
    # The clean manifests travel with the results: they record exactly what was
    # removed from training, so any number here traces back to its data.
    for d in (CODIPAS, COMBINED, MATCHED, YARDSTICK):
        for name in ("clean_manifest.json", "split_manifest.json"):
            if (Path(d) / name).exists():
                add(zf, Path(d) / name, d)

if dest.name == "working":
    shutil.copytree(OUT_ROOT, dest / OUT_ROOT, dirs_exist_ok=True)

print(f"{zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
with zipfile.ZipFile(zip_path) as zf:
    print(f"{len(zf.namelist())} files")
if dest.name == "working":
    print("Find it in the Output pane on the right.")

## What you can and cannot claim afterwards

**Comparable** - two numbers may sit in the same column only if all four hold:

1. from the **yardstick** exam (`data/splits/test.csv`, the same 253 rows),
2. on the **same task** (2, 11 and 10 classes are three different exams),
3. marked `leaked = False`,
4. produced **after the determinism check passed**.

All four are recorded per row, so this is checkable rather than remembered.

**Not comparable, by design:**

- home-exam scores from different corpora - CODIPAS's test set is a different
  exam from Annotated's,
- zero-shot against calibrated yardstick numbers - different questions,
- val against test,
- anything in `results_experiments/` or `results_combined/`, which came from the
  contaminated splits.

**The claims this notebook supports:**

- *Within-dataset* - "trained and tested on X, we get N" (the **home** row).
- *Transfer* - "trained on X, tested on the human-annotated set, we get M"
  (the **yardstick** row). `home - yardstick` is how much was corpus-specific.
- *Reproducible* - "re-running this gives the same number", because section 2
  demonstrated it rather than assuming it.